# marv-hyena 0.2: do the new tools work on the real Evo 2?

Version 0.2.0 added **`gates`** (gate capture, lag-band ablation, gate freezing, channel scaling) and
**`results`** (an append-only results table). The tiny CPU model passes every test, but it only *mirrors*
Vortex. This notebook checks the new tools on the real `evo2_7b`, then uses them for a first look at two
questions the earlier rounds left open.

| step | question | it works if |
|---|---|---|
| 1 | Do the new tools agree with the real model? | all smoke checks pass, including the six new ones |
| 2 | What do Evo 2's gates look like? | gate capture reconstructs every Hyena block |
| 3 | Does LI's help with long-range copying travel through its **far** lags? | band edits run, the model stays healthy, hooks come off |
| 4 | Does **input-dependent gating** matter? | gate freezes run, hooks come off |
| 5 | Is every number in the results table? | one row per measurement, with interval and commit |

**Steps 3 and 4 are exploratory.** They are not pre-registered in `PREDICTIONS.md`, so whatever they show
is a candidate prediction for a later round, not a finding.

**Runtime:** A100 + High-RAM. Leave `QUICK = True` first (a few minutes after the model loads), then set
`QUICK = False` for the full sample sizes.

## 0. Setup

In [ ]:
# GPU check (no torch import yet: the install below may change the torch version)
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# ---- run settings: edit these two lines only ----
QUICK = True      # True = small samples, to check everything runs. False = full sample sizes.
USE_DRIVE = False # False = everything stays on this Colab machine. True = also cache on Google Drive.

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
    os.makedirs(os.environ['HF_HOME'], exist_ok=True)
    OUT_ROOT = '/content/drive/MyDrive/marv_hyena'
else:
    OUT_ROOT = '/content/marv_hyena_out'
OUT = f"{OUT_ROOT}/gates_{'quick' if QUICK else 'full'}"
os.makedirs(OUT, exist_ok=True)
print('mode    =', 'QUICK' if QUICK else 'FULL')
print('outputs =', OUT)

In [ ]:
# Install Evo 2 + marv-hyena (~2-5 min). Same recipe as the round notebooks: no flash-attn (it would compile
# for hours); marv-hyena runs Evo 2's attention through PyTorch's fused kernel instead (marv_hyena/noflash.py).
# evo2 releases declare Python < 3.13 but are pure Python, so force the current one; remove Colab's empty
# transformer-engine package, whose import crashes evo2 (the 7B model needs no TE).
!pip uninstall -y -q transformer-engine transformer_engine
!pip install -q --ignore-requires-python evo2==0.6.0
BRANCH = 'main'
!rm -rf marv-hyena && git clone -q -b {BRANCH} https://github.com/thebnbrkr/marv-hyena.git
!pip install -q -r marv-hyena/requirements.txt

import sys, subprocess, os
sys.path.insert(0, '/content/marv-hyena')
print('commit:', subprocess.run(['git', '-C', 'marv-hyena', 'rev-parse', '--short', 'HEAD'],
                                capture_output=True, text=True).stdout.strip())
for m in ('gates', 'results'):
    assert os.path.exists(f'marv-hyena/marv_hyena/{m}.py'), f'{m}.py missing: this needs marv-hyena 0.2.0+'

In [ ]:
# Must run before anything imports evo2/vortex: lets Vortex import without flash-attn.
from marv_hyena import noflash
noflash.prepare()

import torch, vortex, evo2
from importlib.metadata import version
print('evo2', version('evo2'), '| vtx', version('vtx'), '| torch', torch.__version__)
assert version('evo2') >= '0.6.0', 'old evo2 installed: Runtime -> Disconnect and delete runtime, then rerun'
print('GPU', torch.cuda.get_device_name(0), 'capability', torch.cuda.get_device_capability(0))
assert torch.cuda.get_device_capability(0)[0] >= 8, 'needs an Ampere+ GPU with bf16 (A100 / L4 / H100)'

In [ ]:
# marv-hyena's own tests on the tiny CPU model. Should say "115 passed" (or more).
!cd marv-hyena && python -m pytest -q

In [ ]:
# Data: the annotated E. coli K-12 genome from the evo2 repo
import os, urllib.request
os.makedirs('data', exist_ok=True)
GENOME = 'data/NC_000913.gb'
if not os.path.exists(GENOME):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/ArcInstitute/evo2/main/notebooks/'
                               'sparse_autoencoder/NC_000913.gb', GENOME)

import marv_hyena as mh
from marv_hyena.probes import load_sequence
genome = load_sequence(GENOME)
print(f'E. coli genome: {len(genome):,} letters | marv-hyena {mh.__version__}')
assert mh.__version__ >= '0.2.0'

In [ ]:
# Load Evo 2 7B (downloads ~15 GB the first time)
import time
MODEL_NAME = 'evo2_7b'
t0 = time.time()
hm = mh.HyenaModel.load(MODEL_NAME)
print(f'loaded in {time.time()-t0:.0f}s')
print(hm.describe())

In [ ]:
import json, random
from contextlib import ExitStack
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from marv_hyena.checks import run_smoke_checks

DATASET = 'E. coli K-12 (NC_000913)'
TABLE = mh.ResultsTable(f'{OUT}/table.jsonl')  # one row per measurement, appended as we go
RESULTS = {'mode': 'quick' if QUICK else 'full', 'model': MODEL_NAME}

def jsonable(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)

def save():
    with open(f'{OUT}/results_gates.json', 'w') as fh:
        json.dump(RESULTS, fh, default=jsonable, indent=1)

def boot_mean(vals, n_boot=4000, seed=0):
    '''Mean and 95% percentile bootstrap interval over the replicates in `vals`.'''
    vals = np.asarray(vals, float)
    if len(vals) < 2:
        return float(vals.mean()), (float('nan'), float('nan'))
    r = np.random.default_rng(seed)
    draws = [r.choice(vals, len(vals)).mean() for _ in range(n_boot)]
    return float(vals.mean()), tuple(float(x) for x in np.percentile(draws, [2.5, 97.5]))

def run_with(contexts, ids):
    '''Logits with every context manager in `contexts` active; all hooks removed afterwards.'''
    with ExitStack() as stack:
        for c in contexts:
            stack.enter_context(c)
        return hm.logits(ids)

LI = hm.blocks_of('li')
print('LI blocks:', LI)

## 1. Smoke checks, including the new ones

The six new checks: for one SE, one MR and one LI block, (a) the gate capture reconstructs the real
operator output (x2 × convolution = output), and (b) zeroing a lag band removes **exactly** the band write
that `distance.py` computes independently. Tolerances are bf16-loose (5%). **If anything fails, stop.**

In [ ]:
seq = genome[100_000:104_096]
ok = run_smoke_checks(hm, seq)
RESULTS['smoke_ok'] = ok; save()
assert ok, 'smoke checks failed: the new tools disagree with the installed Vortex -- do not use them'

# hooks from every new tool come off cleanly
ids = hm.ids(seq)
before = hm.logits(ids)
with mh.band_ablate(hm, LI[0], (1, None)):
    hm.logits(ids)
with mh.freeze_gate(hm, LI[0], 'x2'):
    hm.logits(ids)
with mh.scale_channels(hm, LI[0], [0, 1, 2], 0.0):
    hm.logits(ids)
assert torch.equal(before, hm.logits(ids)), 'a hook was left behind'
print('hooks removed: model unchanged after band_ablate, freeze_gate, scale_channels')

## 2. What Evo 2's gates look like

`capture_gates` records the output gate **x2**, the input gate **x1**, v and the convolution at a few
positions of real DNA. Two things per block:

- **rel_error**: x2 × convolution vs the real output. Must be small for every block.
- **closed share**: how often a channel's gate is near zero (under 1% of that channel's typical size).
  A closed gate switches that channel of the operator off at that position.

With a handful of positions this is a **description**, not an estimate.

In [ ]:
POS = list(range(512, len(seq), 512))
caps = mh.capture_gates(hm, ids, POS)
gate_rows = []
for b, g in caps.items():
    gate_rows.append({'block': b, 'kind': g.kind, 'rel_error': g.rel_error,
                      'mean_abs_x2': float(g.x2.abs().mean()), 'mean_abs_x1': float(g.x1.abs().mean()),
                      'closed_x2': float(g.closed_fraction('x2').mean()),
                      'closed_x1': float(g.closed_fraction('x1').mean())})
gates_df = pd.DataFrame(gate_rows)
print(gates_df.round(4).to_string(index=False))
assert gates_df.rel_error.max() < 5e-2, 'gate capture does not reconstruct some block: see rel_error'
RESULTS['gates'] = gate_rows; save()

for r in gate_rows:
    TABLE.add(model=MODEL_NAME, component=(r['block'], 'mixer'), intervention='none',
              task='gate closed share (x2)', dataset=DATASET, n=len(POS), effect=r['closed_x2'],
              unit='share of channels x positions', notes=f"kind={r['kind']} rel_error={r['rel_error']:.1e}")

fig, ax = plt.subplots(figsize=(9, 3))
for kind, colour in zip(('se', 'mr', 'li'), ('tab:blue', 'tab:orange', 'tab:green')):
    sub = gates_df[gates_df.kind == kind]
    ax.bar(sub.block, sub.closed_x2, color=colour, label=kind)
ax.set_xlabel('block'); ax.set_ylabel('closed share of x2'); ax.legend(); ax.set_title('How often output gates are shut')
plt.tight_layout(); plt.show()

## 3. Does LI's help with long-range copying travel through its far lags?

Round 3 found that mean-ablating every LI mixer drops copying across 10,000 letters from 99.7% to 55.6%,
yet the filter reach table says most LI channels look only 4–7 letters back. Those two facts fit together
only if LI's help is **local** (it prepares something attention then uses) or carried by a **few
far-reaching channels**. `band_ablate` can now separate them:

| condition | what is removed, on every LI block |
|---|---|
| baseline | nothing |
| LI mixers ablated | the whole LI write (mean), as in round 3 |
| LI far lags removed | only lags ≥ 1024 of the convolution (mean) |
| LI near lags removed | only lags 1–127 (mean), as a comparison |

**Reading it:** if *far lags removed* falls near *LI mixers ablated*, the far lags carry LI's contribution.
If *near lags removed* does instead, LI helps locally. **Health:** the model's log-probability on ordinary
background DNA (before the insert) should stay close to baseline, otherwise a drop in copying only means
the model broke.

In [ ]:
N_SEEDS = 2 if QUICK else 10
GAPS = (1000, 10000)
LEAD, INSERT = 1000, 200
FAR, NEAR = (1024, None), (1, 127)

MEAN_SEQ = genome[2_000_000:2_008_192]  # LI mean writes come from unrelated DNA
LI_MEANS = mh.mean_writes(hm, hm.ids(MEAN_SEQ), mh.mixers_of(hm, 'li'))
COPY_CONDS = {
    'baseline': lambda: [],
    'LI mixers ablated': lambda: [mh.mean_ablate(hm, LI_MEANS)],
    'LI far lags (1024+) removed': lambda: [mh.band_ablate(hm, b, FAR) for b in LI],
    'LI near lags (1-127) removed': lambda: [mh.band_ablate(hm, b, NEAR) for b in LI],
}

copy_rows = []
t0 = time.time()
for gap in GAPS:
    for s in range(N_SEEDS):
        off = 300_000 + 150_000 * s  # a different genomic site for every replicate
        p = mh.copy_probe(genome[off:off + LEAD + gap + 50], insert_len=INSERT, gap=gap, lead=LEAD, seed=s)
        pids = hm.ids(p.seq)
        for name, make in COPY_CONDS.items():
            logits = run_with(make(), pids)
            sc = mh.score_copy(logits, pids, p, skip=20)
            health = mh.span_logprob(logits, pids, 100, LEAD) / (LEAD - 100)
            copy_rows.append({'gap': gap, 'seed': s, 'condition': name,
                              'second_acc': sc['second_acc'], 'health_lp': health})
print(f'{len(copy_rows)} runs in {time.time() - t0:.0f}s')
RESULTS['copy'] = copy_rows; save()
assert torch.equal(before, hm.logits(ids)), 'a hook was left behind'

In [ ]:
cp = pd.DataFrame(copy_rows)
base_health = cp[cp.condition == 'baseline'].set_index(['gap', 'seed']).health_lp
summary = []
for (gap, cond), sub in cp.groupby(['gap', 'condition'], sort=False):
    acc, ci = boot_mean(sub.second_acc.values)
    dh = sub.set_index(['gap', 'seed']).health_lp - base_health.loc[sub.set_index(['gap', 'seed']).index]
    summary.append({'gap': gap, 'condition': cond, 'n': len(sub), 'copy_acc': acc, 'ci_low': ci[0],
                    'ci_high': ci[1], 'health_change_nats': float(dh.mean())})
    TABLE.add(model=MODEL_NAME, component='all li mixers', intervention=cond, task=f'copy@{gap}',
              dataset=DATASET, n=len(sub), effect=acc, ci=ci, unit='second-copy accuracy',
              notes=f'health change {float(dh.mean()):+.3f} nats/letter', seeds=N_SEEDS)
copy_summary = pd.DataFrame(summary)
print(copy_summary.round(3).to_string(index=False))
RESULTS['copy_summary'] = summary; save()

fig, ax = plt.subplots(figsize=(9, 3.5))
conds = list(COPY_CONDS)
w = 0.8 / len(GAPS)
for i, gap in enumerate(GAPS):
    sub = copy_summary[copy_summary.gap == gap].set_index('condition').loc[conds]
    x = np.arange(len(conds)) + i * w
    ax.bar(x, sub.copy_acc, w, label=f'gap {gap:,}',
           yerr=[sub.copy_acc - sub.ci_low.fillna(sub.copy_acc), sub.ci_high.fillna(sub.copy_acc) - sub.copy_acc])
ax.set_xticks(np.arange(len(conds)) + w * (len(GAPS) - 1) / 2, conds, rotation=15)
ax.set_ylabel('second-copy accuracy'); ax.set_ylim(0, 1.05); ax.legend()
plt.tight_layout(); plt.show()
print('health_change_nats should be near 0 for the band conditions; a large negative value means the model broke')

## 4. Does input-dependent gating matter?

`freeze_gate` replaces a gate with its average over positions, on every channel of every block of one
operator type. Those blocks then apply a **fixed** filter: the convolution still runs, but it no longer
depends on what the DNA says at each position. The measure is how much worse the model predicts ordinary
DNA (mean log-probability per letter, change from baseline; more negative = worse).

**Reading it:** a large drop means that operator type relies on input-dependent gating; a small one means
those blocks behave roughly like fixed filters on this DNA.

In [ ]:
N_WIN = 1 if QUICK else 4
WIN = 8192
rng = random.Random(0)
WINDOWS = [genome[o:o + WIN] for o in sorted(rng.sample(range(10_000, len(genome) - WIN), N_WIN))]
SCORE_FROM = 1024  # skip the start, where the model has little context

GATE_CONDS = {'baseline': lambda: []}
for kind in ('se', 'mr', 'li'):
    for gate in ('x2', 'x1'):
        GATE_CONDS[f'{kind} {gate} frozen'] = lambda k=kind, g=gate: [mh.freeze_gate(hm, b, g) for b in hm.blocks_of(k)]

gate_rows2 = []
t0 = time.time()
for w, win in enumerate(WINDOWS):
    wids = hm.ids(win)
    for name, make in GATE_CONDS.items():
        logits = run_with(make(), wids)
        lp = mh.span_logprob(logits, wids, SCORE_FROM, WIN) / (WIN - SCORE_FROM)
        gate_rows2.append({'window': w, 'condition': name, 'lp': lp})
print(f'{len(gate_rows2)} runs in {time.time() - t0:.0f}s')
assert torch.equal(before, hm.logits(ids)), 'a hook was left behind'

gf = pd.DataFrame(gate_rows2)
base = gf[gf.condition == 'baseline'].set_index('window').lp
freeze_summary = []
for cond, sub in gf[gf.condition != 'baseline'].groupby('condition', sort=False):
    d = (sub.set_index('window').lp - base).values
    m, ci = boot_mean(d)
    kind, gate = cond.split()[:2]
    freeze_summary.append({'condition': cond, 'n_windows': len(d), 'change_nats': m, 'ci_low': ci[0], 'ci_high': ci[1]})
    TABLE.add(model=MODEL_NAME, component=f'all {kind} mixers', intervention=f'freeze_gate {gate}',
              task='next-letter log-prob, change from baseline', dataset=DATASET, n=len(d), effect=m, ci=ci,
              unit='nats/letter')
freeze_df = pd.DataFrame(freeze_summary)
print(freeze_df.round(4).to_string(index=False))
RESULTS['freeze'] = freeze_summary; save()

## 5. The results table

Every number above was appended to one file as it was computed, with its interval, sample size and the
marv-hyena commit that produced it. Later questions become filters over this file. Download it with the
other results (or set `USE_DRIVE = True` to keep it on Drive).

In [ ]:
rows = TABLE.rows()
print(f'{len(rows)} rows in {TABLE.path}')
cols = ['component', 'intervention', 'task', 'n', 'effect', 'ci_low', 'ci_high', 'unit', 'commit']
print(pd.DataFrame(rows)[cols].round(3).to_string(index=False))
print('\nexample query -- every copy@10000 result:')
fmt = lambda x: 'n/a' if x is None else f'{x:.3f}'
for r in TABLE.where(task='copy@10000'):
    print(f"  {r['intervention']:<30} {fmt(r['effect'])}  [{fmt(r['ci_low'])}, {fmt(r['ci_high'])}]")
save()
if not USE_DRIVE:
    try:
        from google.colab import files
        files.download(str(TABLE.path)); files.download(f'{OUT}/results_gates.json')
    except ImportError:
        pass